# 📘 통계적 추정

**통계적 추정**(Estimation)은 표본 데이터로부터 모수(모평균, 모분산 등)를 추측하는 방법입니다.

추정에는 두 가지가 있습니다:
- **점추정**: 모수를 하나의 값으로 추정 (예: 표본평균 ≈ 모평균)
- **구간추정**: 모수가 포함될 범위를 확률과 함께 추정 (예: 95% 신뢰구간)

**학습 목표:**
- 점추정과 구간추정의 개념
- 신뢰구간의 계산과 해석
- 신뢰구간의 폭을 결정하는 요소
- 신뢰구간이 모평균을 포함하는 확률 시뮬레이션

## 1. 분석 준비 — 데이터 불러오기

물고기 길이 데이터를 불러옵니다. 이 데이터는 10마리의 물고기 길이 측정값입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  라이브러리 임포트 + 데이터 로드            │
# └─────────────────────────────────────────┘

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 물고기 길이 데이터 불러오기
fish = pd.read_csv("fish_length.csv")["length"]
print("물고기 길이 데이터:")
print(fish)
print(f"\n표본 크기: {len(fish)}")
print(f"표본평균: {np.mean(fish):.4f}")
print(f"표본표준편차: {np.std(fish, ddof=1):.4f}")

## 2. 점추정

**점추정**(Point Estimation)은 모수를 하나의 값으로 추정하는 것입니다.

| 모수 | 점추정량 |
|------|---------|
| 모평균 μ | 표본평균 x̄ |
| 모분산 σ² | 불편분산 s² |

> 💡 점추정은 간단하지만, "얼마나 정확한지"를 알 수 없다는 한계가 있습니다.
> 그래서 구간추정이 필요합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  점추정 — 모평균과 모분산               │
# │  표본평균 → 모평균 추정                │
# │  불편분산 → 모분산 추정                │
# └─────────────────────────────────────────┘

# 모평균의 점추정: 표본평균
mu_hat = np.mean(fish)
print(f"모평균의 점추정 (표본평균): {mu_hat:.4f}")

# 모분산의 점추정: 불편분산
sigma2_hat = np.var(fish, ddof=1)
print(f"모분산의 점추정 (불편분산): {sigma2_hat:.4f}")

# 모표준편차의 점추정
sigma_hat = np.std(fish, ddof=1)
print(f"모표준편차의 점추정: {sigma_hat:.4f}")

print(f"\n→ 점추정은 하나의 값이므로 정확도를 알 수 없음")
print(f"→ 구간추정으로 신뢰성을 표현 필요")

## 3. 구간추정 — 신뢰구간

**구간추정**(Interval Estimation)은 모수가 포함될 범위를 확률과 함께 추정합니다.

**95% 신뢰구간**: 100번 표본을 추출해 신뢰구간을 구하면, 약 95번은 모평균을 포함합니다.

$$\bar{x} - t_{0.025} \times SE \leq \mu \leq \bar{x} + t_{0.025} \times SE$$

여기서:
- x̄: 표본평균
- t: t분포의 임계값 (자유도 = n-1)
- SE: 표준오차 = s/√n

In [ ]:
# ┌─────────────────────────────────────────┐
# │  95% 신뢰구간 계산                       │
# │  stats.t.interval()로 간편하게 계산    │
# │  자유도 = n - 1                         │
# └─────────────────────────────────────────┘

# 자유도
df = len(fish) - 1
print(f"자유도: {df}")

# 표준오차
se = sigma_hat / np.sqrt(len(fish))
print(f"표준오차 SE: {se:.4f}")
print(f"  = {sigma_hat:.4f} / √{len(fish)}")
print(f"  = {sigma_hat:.4f} / {np.sqrt(len(fish)):.4f}")

# 95% 신뢰구간
interval = stats.t.interval(alpha=0.95, df=df, loc=mu_hat, scale=se)
print(f"\n95% 신뢰구간: [{interval[0]:.4f}, {interval[1]:.4f}]")
print(f"\n→ 모평균 μ가 [{interval[0]:.2f}, {interval[1]:.2f}] 사이에 있을 확률이 95%")

## 4. 신뢰구간의 상세 계산

`stats.t.interval()`이 내부적으로 어떻게 계산되는지 이해해 봅시다.

1. t분포의 97.5% 분위수(t_0.975)를 구함
2. 하한 = x̄ - t × SE, 상한 = x̄ + t × SE

> 💡 이 계산 과정을 이해하면 신뢰구간의 의미를 더 깊이 알 수 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  신뢰구간 상세 계산                     │
# │  t분위수 → 하한/상한 직접 계산          │
# └─────────────────────────────────────────┘

# t분포의 97.5% 분위수 (상위 2.5%의 역)
t_975 = stats.t.ppf(q=0.975, df=df)
print(f"t분포 97.5% 분위수: {t_975:.4f}")

# 하한과 상한 직접 계산
lower = mu_hat - t_975 * se
upper = mu_hat + t_975 * se
print(f"\n하한: {mu_hat:.4f} - {t_975:.4f} × {se:.4f} = {lower:.4f}")
print(f"상한: {mu_hat:.4f} + {t_975:.4f} × {se:.4f} = {upper:.4f}")
print(f"\n직접 계산: [{lower:.4f}, {upper:.4f}]")
print(f"interval():  [{interval[0]:.4f}, {interval[1]:.4f}]")
print(f"→ 두 결과가 일치함을 확인")

## 5. 신뢰구간의 폭을 결정하는 요소

신뢰구간의 폭은 세 가지 요소에 의해 결정됩니다:

| 요소 | 폭에 미치는 영향 |
|------|----------------|
| 표본표준편차 ↑ | 폭이 넓어짐 (데이터가 퍼져 있음) |
| 표본 크기 ↑ | 폭이 좁아짐 (정밀도 향상) |
| 신뢰수준 ↑ | 폭이 넓어짐 (99% > 95%) |

> 💡 표본 크기를 늘리면 신뢰구간이 좁아집니다 → 더 정밀한 추정이 가능!

In [ ]:
# ┌─────────────────────────────────────────┐
# │  신뢰구간 폭에 영향을 미치는 요소        │
# │  1) 표본표준편차가 크면 → 폭이 넓어짐   │
# │  2) 표본 크기가 크면 → 폭이 좁아짐      │
# │  3) 신뢰수준이 높으면 → 폭이 넓어짐     │
# └─────────────────────────────────────────┘

# 1) 표본표준편차가 10배 큰 경우
se2 = (sigma_hat * 10) / np.sqrt(len(fish))
interval_sd10 = stats.t.interval(alpha=0.95, df=df, loc=mu_hat, scale=se2)
print("=== 표본표준편차가 10배 큰 경우 ===")
print(f"95% 신뢰구간: [{interval_sd10[0]:.4f}, {interval_sd10[1]:.4f}]")
print(f"폭: {interval_sd10[1] - interval_sd10[0]:.4f}")
print(f"→ 표준편차가 크면 신뢰구간이 넓어짐")

# 2) 표본 크기가 10배 큰 경우
df2 = (len(fish) * 10) - 1
se3 = sigma_hat / np.sqrt(len(fish) * 10)
interval_n10 = stats.t.interval(alpha=0.95, df=df2, loc=mu_hat, scale=se3)
print(f"\n=== 표본 크기가 10배 큰 경우 ===")
print(f"95% 신뢰구간: [{interval_n10[0]:.4f}, {interval_n10[1]:.4f}]")
print(f"폭: {interval_n10[1] - interval_n10[0]:.4f}")
print(f"→ 표본 크기가 크면 신뢰구간이 좁아짐")

# 3) 99% 신뢰구간
interval_99 = stats.t.interval(alpha=0.99, df=df, loc=mu_hat, scale=se)
print(f"\n=== 99% 신뢰구간 ===")
print(f"99% 신뢰구간: [{interval_99[0]:.4f}, {interval_99[1]:.4f}]")
print(f"95% 신뢰구간: [{interval[0]:.4f}, {interval[1]:.4f}]")
print(f"→ 신뢰수준이 높으면 신뢰구간이 넓어짐")

## 6. 신뢰구간의 해석 — 시뮬레이션

**"95% 신뢰구간"이란 무슨 뜻일까요?**

→ 모평균(4)을 포함하는 신뢰구간이 95% 비율로 나타난다는 뜻입니다.

20,000번 시뮬레이션으로 확인해 봅시다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  신뢰구간 시뮬레이션                    │
# │  95% 신뢰구간이 모평균을 포함하는 비율  │
# │  20,000회 반복으로 확인                │
# └─────────────────────────────────────────┘

np.random.seed(1)
norm_dist = stats.norm(loc=4, scale=0.8)
n_sim = 20000
be_included = np.zeros(n_sim, dtype=bool)

for i in range(n_sim):
    sample = norm_dist.rvs(size=10)
    x_bar = np.mean(sample)
    s = np.std(sample, ddof=1)
    se = s / np.sqrt(10)
    df = 9
    interval = stats.t.interval(alpha=0.95, df=df, loc=x_bar, scale=se)
    be_included[i] = (interval[0] <= 4) and (interval[1] >= 4)

coverage = np.sum(be_included) / len(be_included)
print(f"시뮬레이션 횟수: {n_sim:,}")
print(f"모평균(4)을 포함한 신뢰구간 비율: {coverage:.4f}")
print(f"이론적 신뢰수준: 0.9500")
print(f"\n→ 95% 신뢰구간이 모평균을 포함할 확률이 약 95%임을 확인")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  신뢰구간 시각화 — 50개 표본의 신뢰구간   │
# │  모평균(빨간 점선)을 포함하면 파란색    │
# │  포함하지 못하면 빨간색               │
# └─────────────────────────────────────────┘

np.random.seed(42)
n_vis = 50
intervals = []
includes = []

for i in range(n_vis):
    sample = norm_dist.rvs(size=10)
    x_bar = np.mean(sample)
    s = np.std(sample, ddof=1)
    se = s / np.sqrt(10)
    ci = stats.t.interval(alpha=0.95, df=9, loc=x_bar, scale=se)
    intervals.append(ci)
    includes.append(ci[0] <= 4 <= ci[1])

fig, ax = plt.subplots(figsize=(8, 8))
for i, (ci, inc) in enumerate(zip(intervals, includes)):
    color = 'steelblue' if inc else 'red'
    ax.plot([ci[0], ci[1]], [i, i], color=color, linewidth=1.5)
    ax.plot(x_bar, i, 'o', color=color, markersize=3)

ax.axvline(4, color='red', linestyle='--', linewidth=2, label='모평균 μ=4')
ax.set_xlabel('물고기 길이')
ax.set_ylabel('표본 번호')
ax.set_title('95% 신뢰구간 시뮬레이션 (50개 표본)')
ax.legend()
plt.tight_layout()
plt.savefig('confidence_interval_sim.png', dpi=100)
plt.show()

n_included = sum(includes)
print(f"모평균을 포함한 신뢰구간: {n_included}/{n_vis} ({n_included/n_vis*100:.0f}%)")
print(f"모평균을 벗어난 신뢰구간: {n_vis - n_included}/{n_vis} ({(n_vis-n_included)/n_vis*100:.0f}%)")
print(f"→ 약 95%의 신뢰구간이 모평균을 포함함")

## 🎯 연습 문제

1. `fish_length.csv` 데이터에서 90% 신뢰구간을 계산하고, 95% 신뢰구간과 폭을 비교하세요.
2. 표본 크기를 2배, 5배, 10배로 늘렸을 때 신뢰구간의 폭이 어떻게 변하는지 계산하세요.
3. 99% 신뢰수준으로 10,000번 시뮬레이션하여 모평균 포함 비율을 확인하세요.
4. 신뢰구간 시뮬레이션에서 90%, 95%, 99% 신뢰수준별 포함 비율을 비교하세요.